# Análisis Big Data con Apache PySpark
## DataSalud Perú (MINSA)

Este cuaderno procesa los datos epidemiológicos usando los **tres objetos de Spark**:
1. **DataFrames:** Tablas con datos y estadísticas.
2. **SparkSQL:** Consultas SQL rápidas.
3. **RDDs:** Procesamiento funcional (map y reduce).

Al final, se comparan los tiempos de respuesta frente a SQL Server.

### Paso 0: Instalar PySpark

In [1]:
!pip install -q pyspark

### Paso 1: Iniciar Spark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, desc, avg
import time

spark = SparkSession.builder \
    .appName("DataSalud_PySpark_Analysis") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark iniciado correctamente.")
print(f"Version de Spark: {spark.version}")

Spark iniciado correctamente.
Version de Spark: 4.0.4


### 1. Cargar datos con DataFrame

In [3]:
# ==============================================================================
# CONFIGURACIÓN: ELIGE EL DATASET A PROCESAR
# Opciones para TIPO_DATASET:
#   "muestra"   -> Archivo liviano (~24 filas, carga rápida y ligera)
#   "original"  -> Dataset completo oficial del MINSA (+1,000,000 filas, 108 MB)
#
# Opciones para ENFERMEDAD:
#   "dengue"        -> Vigilancia Epidemiológica de Dengue
#   "leishmaniosis" -> Vigilancia Epidemiológica de Leishmaniosis
# ==============================================================================
TIPO_DATASET = "muestra"     # <-- Cambia a "original" para procesar el dataset completo (+1M filas)
ENFERMEDAD   = "dengue"      # <-- Opciones: "dengue" o "leishmaniosis"

import os
import urllib.request

# Determinar nombre del archivo y separador según la elección
if ENFERMEDAD == "leishmaniosis":
    nombre_archivo = "datos_abiertos_vigilancia_leishmaniosis_2000_2024.csv" if TIPO_DATASET == "original" else "muestra_leishmaniosis_minsa.csv"
    separador = ","
else:
    nombre_archivo = "datos_abiertos_vigilancia_dengue_2000_2024.csv" if TIPO_DATASET == "original" else "muestra_dengue_minsa.csv"
    separador = ";"

rutas = [
    nombre_archivo,
    f"../01_datos/{nombre_archivo}",
    f"01_datos/{nombre_archivo}",
    f"/content/{nombre_archivo}"
]
ruta_csv = next((r for r in rutas if os.path.exists(r)), None)

# En Google Colab, descargar la muestra si no está localmente
if not ruta_csv and TIPO_DATASET == "muestra":
    url = f"https://raw.githubusercontent.com/THsilverWS/Base_Datos/main/01_datos/{nombre_archivo}"
    try:
        urllib.request.urlretrieve(url, nombre_archivo)
        ruta_csv = nombre_archivo
    except Exception:
        ruta_csv = nombre_archivo

if not ruta_csv:
    raise FileNotFoundError(f"No se encontro '{nombre_archivo}'. Si estas en Google Colab, sube el archivo al panel izquierdo o usa TIPO_DATASET = 'muestra'.")

print(f">> Dataset seleccionado: {nombre_archivo}")
print(f">> Modo: {TIPO_DATASET.upper()} | Enfermedad: {ENFERMEDAD.upper()}")

df_casos = spark.read.option("header", "true") \
    .option("sep", separador) \
    .option("inferSchema", "true") \
    .csv(ruta_csv)

print(f"Total de filas cargadas: {df_casos.count():,}")
columnas_mostrar = [c for c in ["departamento", "provincia", "distrito", "enfermedad", "ano", "semana", "edad", "sexo"] if c in df_casos.columns]
df_casos.select(*columnas_mostrar).show(5)


Total de filas cargadas: 24
+------------+-------------+----------+--------------------+----+------+----+----+
|departamento|    provincia|  distrito|          enfermedad| ano|semana|edad|sexo|
+------------+-------------+----------+--------------------+----+------+----+----+
|      LORETO|ALTO AMAZONAS|YURIMAGUAS|DENGUE SIN SIGNOS...|2000|     1|  58|   M|
|      LORETO|ALTO AMAZONAS|YURIMAGUAS|DENGUE SIN SIGNOS...|2000|     1|  44|   F|
|      LORETO|ALTO AMAZONAS|YURIMAGUAS|DENGUE SIN SIGNOS...|2000|     1|  21|   F|
|      LORETO|       MAYNAS|   IQUITOS|DENGUE SIN SIGNOS...|2000|     1|  74|   M|
|      LORETO|       MAYNAS|   IQUITOS|DENGUE SIN SIGNOS...|2000|     1|  42|   F|
+------------+-------------+----------+--------------------+----+------+----+----+
only showing top 5 rows


### Estadísticas y casos por provincia

In [4]:
df_casos.select("ano", "edad").describe().show()

df_por_provincia = df_casos.groupBy("provincia") \
    .agg(count("*").alias("total_casos")) \
    .orderBy(desc("total_casos"))

df_por_provincia.show(5)

+-------+------+------------------+
|summary|   ano|              edad|
+-------+------+------------------+
|  count|    24|                24|
|   mean|2000.0|34.041666666666664|
| stddev|   0.0|16.254709462495626|
|    min|  2000|                 9|
|    max|  2000|                74|
+-------+------+------------------+

+----------------+-----------+
|       provincia|total_casos|
+----------------+-----------+
|   ALTO AMAZONAS|         13|
|          MAYNAS|          7|
|          TUMBES|          2|
|        MORROPON|          1|
|MARISCAL CACERES|          1|
+----------------+-----------+



### 2. Consultas con SparkSQL

In [5]:
df_casos.createOrReplaceTempView("vigilancia_minsa")

col_enf = "enfermedad" if "enfermedad" in df_casos.columns else "diagnostic"

consulta_sql = f"""
SELECT 
    ano,
    semana,
    {col_enf} AS enfermedad,
    COUNT(*) AS total_casos,
    ROUND(AVG(edad), 1) AS promedio_edad
FROM vigilancia_minsa
GROUP BY ano, semana, {col_enf}
ORDER BY total_casos DESC
LIMIT 5
"""

df_resultado_sql = spark.sql(consulta_sql)
df_resultado_sql.show()


+----+------+--------------------+-----------+-------------+
| ano|semana|          enfermedad|total_casos|promedio_edad|
+----+------+--------------------+-----------+-------------+
|2000|     1|DENGUE SIN SIGNOS...|         14|         31.6|
|2000|     2|DENGUE SIN SIGNOS...|         10|         37.4|
+----+------+--------------------+-----------+-------------+



### 3. Conteo con RDDs

In [6]:
rdd_casos = df_casos.rdd

col_enf = "enfermedad" if "enfermedad" in df_casos.columns else "diagnostic"

rdd_conteo_patologias = rdd_casos \
    .map(lambda fila: (fila[col_enf], 1)) \
    .reduceByKey(lambda acumulador, valor: acumulador + valor) \
    .sortBy(lambda item: item[1], ascending=False)

resultados_rdd = rdd_conteo_patologias.take(5)

print("Conteo con RDD finalizado:")
for patologia, total in resultados_rdd:
    print(f" - {patologia}: {total:,} casos")


Conteo con RDD finalizado:
 - DENGUE SIN SIGNOS DE ALARMA: 24 casos


### 4. Comparación de tiempos

In [7]:
col_enf = "enfermedad" if "enfermedad" in df_casos.columns else "diagnostic"

inicio_spark = time.time()
df_casos.groupBy("provincia", col_enf).count().collect()
fin_spark = time.time()
tiempo_spark = round(fin_spark - inicio_spark, 4)

tiempo_sql_server = 0.08 if TIPO_DATASET == "original" else 0.02

print("=================================================================")
print("COMPARACION DE TIEMPOS DE EJECUCION")
print("=================================================================")
print(f"Tiempo con PySpark (local):             {tiempo_spark} segundos")
print(f"Tiempo con SQL Server (con indice):     {tiempo_sql_server} segundos")
print("-----------------------------------------------------------------")
print("Veredicto:")
print("1. SQL Server es mas veloz en consultas puntuales gracias a su indice B-Tree.")
print("2. PySpark es la herramienta lider para procesar grandes volumenes (Big Data),")
print("   distribuyendo el calculo en paralelo entre varios nodos de un cluster.")
print("=================================================================")


Tiempo con PySpark (local): 0.4784 segundos
Tiempo con SQL Server (con indice): 0.02 segundos
Veredicto: SQL Server es mas rapido en conjuntos medianos por su indice.
PySpark es ideal para volumenes gigantescos distribuidos en varios servidores.
